In [5]:
import pandas as pd

sample = pd.read_csv("../data/raw/sample_submission.csv")
mine   = pd.read_csv("submission_v4.csv")
submission = pd.read_csv("submission_v4.csv")
# riordina le label secondo l'ordine del sample
mine = mine.set_index("Id").loc[sample["Id"]].reset_index()

mine.to_csv("submission_final.csv", index=False)

In [13]:
submission.head()

,Id,Predicted
0,0,5.0
1,1,2.0
2,2,5.0
3,3,NaN
4,4,5.0


In [3]:
df_eval = pd.read_csv("../data/raw/evaluation.csv")

In [8]:
submission.head()

,Id,Predicted
0,0,5.0
1,1,2.0
2,2,5.0
3,3,NaN
4,4,5.0


In [11]:
submission.head()

,Id,Predicted
0,0,5.0
1,1,2.0
2,2,5.0
3,3,NaN
4,4,5.0


In [12]:
mine   = pd.read_csv("submission_v4.csv")

missing = set(sample["Id"]) - set(mine["Id"])
extra   = set(mine["Id"]) - set(sample["Id"])

print("Missing Id in mine:", len(missing))
print("Extra Id in mine:", len(extra))

list(missing)[:10]

Missing Id in mine: 0
Extra Id in mine: 0


[]

In [14]:
eval["Id"] == "3"

TypeError: 'builtin_function_or_method' object is not subscriptable

In [16]:
submission["Id"] == "3"

0        False
1        False
2        False
3        False
4        False
         ...  
19995    False
19996    False
19997    False
19998    False
19999    False
Name: Id, Length: 20000, dtype: bool

In [17]:
import pandas as pd
import numpy as np

df_dev  = pd.read_csv("../data/raw/development.csv")
df_eval = pd.read_csv("../data/raw/evaluation.csv")

df_dev["source"] = df_dev["source"].fillna("UNK").astype(str)
df_eval["source"] = df_eval["source"].fillna("UNK").astype(str)

# label più frequente globale (fallback)
global_label = df_dev["label"].mode()[0]

# mappa source -> label più frequente
source_to_label = (
    df_dev
    .groupby("source")["label"]
    .agg(lambda x: x.value_counts().idxmax())
    .to_dict()
)


In [18]:
predicted = [
    source_to_label.get(src, global_label)
    for src in df_eval["source"].values
]

predicted = np.asarray(predicted, dtype=int)


In [19]:
submission = pd.DataFrame({
    "Id": df_eval["Id"].values,
    "Predicted": predicted
})

# CHECK CRITICI
assert submission.isna().sum().sum() == 0
assert len(submission) == len(df_eval)
assert submission["Predicted"].between(0, 6).all()

submission.to_csv("submission_source_only.csv", index=False)
